# 第 15 章 · A2A Server 的装配（书里的片段）

这一格摘自 `a2a-samples` 仓库的 `birthday_planner_adk` 例子，**在本机跑不了**，原因有三：

1. `CalendarToolset` 需要 Google OAuth（`GOOGLE_CLIENT_ID` / `GOOGLE_CLIENT_SECRET`）
2. 三个单元都缺 import：`os`、`AgentSkill`、`AgentCard`、`Runner`、`ADKAgentExecutor`、
   `DefaultRequestHandler`、`InMemoryTaskStore` 都未定义
3. 第二格在 `request_handler = ...` 处截断，后面还有 `uvicorn.run(...)` 等未给出

它想演示的是 A2A Server 的四步装配：

```python
skill         = AgentSkill(...)                    # 声明能力
agent_card    = AgentCard(..., skills=[skill])     # 数字身份证
agent_executor = ADKAgentExecutor(runner, card)    # 把 ADK agent 接进 A2A
request_handler = DefaultRequestHandler(...)       # JSON-RPC 请求处理
```

**这四步现在被 ADK 收进了一个函数**：

```python
app = to_a2a(root_agent, host="127.0.0.1", port=9999)
```

Agent Card 从 agent 的 `name` / `description` / 工具签名自动生成。

可运行版本见 `../../ch15_a2a/`：
- `agent.py`   —— 两个本地工具（`add` / `now`）替代 Calendar
- `server.py`  —— `to_a2a()` + uvicorn，暴露 `/.well-known/agent-card.json` 与 JSON-RPC
- `client.py`  —— 发现 Card → 同步调用 → 流式调用

实测：Agent Card 自动列出了 3 个 skill；同步调用返回 1 个事件（task COMPLETED），
流式调用返回 7 个事件（SUBMITTED → WORKING×4 → artifact → COMPLETED）。


In [ ]:
# Code Snippet

import datetime

from google.adk.agents import LlmAgent # type: ignore[import-untyped]
from google.adk.tools.google_api_tool import CalendarToolset # type: ignore[import-untyped]


async def create_agent(client_id, client_secret) -> LlmAgent:
    """Constructs the ADK agent."""
    toolset = CalendarToolset(client_id=client_id, client_secret=client_secret)
    return LlmAgent(
        model='gemini-2.0-flash-001',
        name='calendar_agent',
        description="An agent that can help manage a user's calendar",
        instruction=f"""
You are an agent that can help manage a user's calendar.

Users will request information about the state of their calendar or to make changes to
their calendar. Use the provided tools for interacting with the calendar API.

If not specified, assume the calendar the user wants is the 'primary' calendar.

When using the Calendar API tools, use well-formed RFC3339 timestamps.

Today is {datetime.datetime.now()}.
""",
        tools=await toolset.get_tools(),
    )

In [ ]:
#Code Snippet

def main(host: str, port: int):
    # Verify an API key is set.
    # Not required if using Vertex AI APIs.
    if os.getenv('GOOGLE_GENAI_USE_VERTEXAI') != 'TRUE' and not os.getenv(
        'GOOGLE_API_KEY'
    ):
        raise ValueError(
            'GOOGLE_API_KEY environment variable not set and '
            'GOOGLE_GENAI_USE_VERTEXAI is not TRUE.'
        )

    skill = AgentSkill(
        id='check_availability',
        name='Check Availability',
        description="Checks a user's availability for a time using their Google Calendar",
        tags=['calendar'],
        examples=['Am I free from 10am to 11am tomorrow?'],
    )

    agent_card = AgentCard(
        name='Calendar Agent',
        description="An agent that can manage a user's calendar",
        url=f'http://{host}:{port}/',
        version='1.0.0',
        defaultInputModes=['text'],
        defaultOutputModes=['text'],
        capabilities=AgentCapabilities(streaming=True),
        skills=[skill],
    )

    adk_agent = asyncio.run(create_agent(
        client_id=os.getenv('GOOGLE_CLIENT_ID'),
        client_secret=os.getenv('GOOGLE_CLIENT_SECRET'),
    ))
    runner = Runner(
        app_name=agent_card.name,
        agent=adk_agent,
        artifact_service=InMemoryArtifactService(),
        session_service=InMemorySessionService(),
        memory_service=InMemoryMemoryService(),
    )
    agent_executor = ADKAgentExecutor(runner, agent_card)

    async def handle_auth(request: Request) -> PlainTextResponse:
        await agent_executor.on_auth_callback(
            str(request.query_params.get('state')), str(request.url)
        )
        return PlainTextResponse('Authentication successful.')

    request_handler = DefaultRequestHandler(
        agent_executor=agent_executor, task_store=InMemoryTaskStore()
    )

    a2a_app = A2AStarletteApplication(
        agent_card=agent_card, http_handler=request_handler
    )
    routes = a2a_app.routes()
    routes.append(
        Route(
            path='/authenticate',
            methods=['GET'],
            endpoint=handle_auth,
        )
    )
    app = Starlette(routes=routes)

    uvicorn.run(app, host=host, port=port)


if __name__ == '__main__':
    main()